# Lab 1 · 문장을 딥러닝이 읽을 수 있는 숫자로 바꾸기 (Subword 전처리)

**오늘 배우는 것 (실습 20분)**
1. 문장을 토큰(subword)으로 쪼개기
2. 문장마다 길이가 다른 문제를 `MAX_LEN`으로 맞추기
3. 토큰을 숫자 ID로 바꾸고, train/val/test로 나눠서 저장하기

이렇게 저장한 파일은 다음 시간(MLP, RNN, LSTM 등)에서 그대로 불러와 씁니다.

In [1]:
!pip install -q transformers scikit-learn openpyxl

In [2]:
import numpy as np
import pandas as pd
from transformers import AutoTokenizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

## 1. 데이터 불러오기

In [3]:
url = "https://raw.githubusercontent.com/hongsukyi/Lectures/main/data/aihub_conver.xlsx"
df = pd.read_excel(url, engine="openpyxl")

print("전체 문장 수:", len(df))
df.head()

전체 문장 수: 100000


,대분류,소분류,상황,Set Nr.,발화자,원문,번역문
0,비즈니스,회의,의견 교환하기,1,A-1,이번 신제품 출시에 대한 시장의 반응은 어떤가요?,How is the market's reaction to the newly rele...
1,비즈니스,회의,의견 교환하기,1,B-1,판매량이 지난번 제품보다 빠르게 늘고 있습니다.,The sales increase is faster than the previous...
2,비즈니스,회의,의견 교환하기,1,A-2,그렇다면 공장에 연락해서 주문량을 더 늘려야겠네요.,"Then, we'll have to call the manufacturer and ..."
3,비즈니스,회의,의견 교환하기,1,B-2,"네, 제가 연락해서 주문량을 2배로 늘리겠습니다.","Sure, I'll make a call and double the volume o..."
4,비즈니스,회의,의견 교환하기,2,A-1,지난 회의 마지막에 논의했던 안건을 다시 볼까요?,Shall we take a look at the issues we discusse...


## 2. 3개 범주만 골라서 균형 있게 뽑기
전체 범주가 너무 많아서(2000개 넘음), 오늘은 **면접 / 학교 / 협상** 3개만 사용합니다.

In [4]:
label_map = {
    "취직 면접 상황": "면접",
    "학교생활 (시험, 졸업, 입학 등)": "학교",
    "제안 및 협상하기": "협상",
}

df["label"] = df["상황"].map(label_map)
df = df.dropna(subset=["원문", "label"])

print(df["label"].value_counts())

label
면접    1032
학교     976
협상     924
Name: count, dtype: int64


In [5]:
N = 800  # 범주마다 800문장씩 (3개 x 800 = 2400문장)

df = df.groupby("label").sample(n=N, random_state=SEED)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)  # 순서 섞기

df = df[["원문", "label"]].rename(columns={"원문": "text"})
print("최종 문장 수:", len(df))
df.head()

최종 문장 수: 2400


,text,label
0,계약 기간이 만료되면 정규직으로 전환이 되나요?,협상
1,너무 좋은 생각입니다. 서로 생각할 시간이 조금 필요한 것 같아요.,협상
2,"토론 동아리를 알아보고 있는데, 괜찮은 데 알아?",학교
3,임원진뿐만 아니라 팀원들의 의견도 들어봐야 하는 거 아닌가요?,협상
4,전문 컨설턴트를 고용해 인력을 관리하는 방법에 대해 조언을 얻어야 할 것 같아요.,협상


## 3. 토크나이저 불러오기
문장을 의미 단위(subword)로 쪼개는 도구입니다. 한국어용 `klue/bert-base`를 씁니다.

In [6]:
tok = AutoTokenizer.from_pretrained("klue/bert-base")

vocab_size = tok.vocab_size
print("vocab_size:", vocab_size)  # 다음 시간 Embedding 층 만들 때 필요한 값

sample = df["text"][0]
print("\n예문:", sample)
print("토큰:", tok.tokenize(sample))

vocab_size: 32000

예문: 계약 기간이 만료되면 정규직으로 전환이 되나요?
토큰: ['계약', '기간', '##이', '만료', '##되', '##면', '정규직', '##으로', '전환', '##이', '되', '##나', '##요', '?']


## 4. MAX_LEN 정하기
모든 문장 길이를 똑같이 맞춰야 딥러닝 모델에 넣을 수 있습니다.
너무 길게 잡으면 낭비, 너무 짧으면 문장이 잘립니다. **문장 길이의 95%를 담을 수 있는 길이**로 정합니다.

In [7]:
lens = [len(tok.tokenize(t)) for t in df["text"]]

max_len = int(np.percentile(lens, 95)) + 2  # +2는 [CLS], [SEP] 자리
max_len = ((max_len + 7) // 8) * 8           # 8의 배수로 올림 (이미 8의 배수면 그대로 유지)

print("문장 길이 95%:", int(np.percentile(lens, 95)))
print("MAX_LEN:", max_len)  # 다음 시간 Embedding 층 만들 때 필요한 값

문장 길이 95%: 25
MAX_LEN: 32


## 5. 문장을 숫자 ID로 바꾸기 (토큰화)

In [8]:
out = tok(
    df["text"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=max_len,
)

ids = np.array(out["input_ids"])
mask = np.array(out["attention_mask"])

print("ids shape:", ids.shape)   # (문장 수, MAX_LEN)
print("예시 ids[0]:", ids[0][:10], "...")
print("예시 mask[0]:", mask[0][:10], "...")

ids shape: (2400, 32)
예시 ids[0]: [    2  4123  3960  2052 12664  2496  2460  8204  6233  4561] ...
예시 mask[0]: [1 1 1 1 1 1 1 1 1 1] ...


## 6. 라벨을 숫자로 바꾸기

In [9]:
le = LabelEncoder()
y = le.fit_transform(df["label"])
classes = le.classes_.tolist()

print("classes:", classes)   # 0, 1, 2가 각각 어떤 범주인지
print("y[:10]:", y[:10])

classes: ['면접', '학교', '협상']
y[:10]: [2 2 1 2 2 0 1 2 1 1]


## 7. train / val / test 나누기
학습용(70%) / 검증용(15%) / 평가용(15%)으로 나눕니다.
**다음 시간부터 모든 모델이 이 나눔을 그대로 재사용**하므로, 여기서 딱 한 번만 나눕니다.

In [10]:
idx = np.arange(len(y))

idx_train, idx_tmp = train_test_split(idx, test_size=0.3, random_state=SEED, stratify=y)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.5, random_state=SEED, stratify=y[idx_tmp])

print("train:", len(idx_train), " val:", len(idx_val), " test:", len(idx_test))

train: 1680  val: 360  test: 360


## 8. 파일로 저장

In [11]:
fname = "data.npz"

np.savez(
    fname,
    input_ids=ids,
    attention_mask=mask,
    labels=y,
    texts=df["text"].to_numpy(dtype=object),
    train_idx=idx_train,
    val_idx=idx_val,
    test_idx=idx_test,
    vocab_size=np.int32(vocab_size),
    max_len=np.int32(max_len),
    pad_id=np.int32(tok.pad_token_id),
    num_classes=np.int32(len(classes)),
    class_names=np.array(classes),
)

print("저장 완료:", fname)

저장 완료: data.npz


## 9. 잘 저장됐는지 확인

In [12]:
check = np.load(fname, allow_pickle=True)

print("input_ids shape:", check["input_ids"].shape)
print("train/val/test:", len(check["train_idx"]), len(check["val_idx"]), len(check["test_idx"]))
print("class_names:", check["class_names"])

input_ids shape: (2400, 32)
train/val/test: 1680 360 360
class_names: ['면접' '학교' '협상']


## 10. 다음 실습을 위해 파일 내려받기

Colab은 세션(런타임)이 끊기면 저장한 파일이 사라집니다. Lab2·Lab3를 **다른 시간에 새 Colab 세션**에서 이어서 하려면, 지금 만든 npz 파일을 내 컴퓨터로 내려받아 두세요.

In [13]:
try:
    from google.colab import files
    files.download(fname)
except ImportError:
    print(f"Colab이 아닌 환경입니다. '{fname}' 파일이 현재 작업 폴더에 이미 저장되어 있습니다.")


Colab이 아닌 환경입니다. 'data.npz' 파일이 현재 작업 폴더에 이미 저장되어 있습니다.


Lab2·Lab3를 시작할 때는 이 파일을 다시 업로드하면 됩니다.

> **같은 세션에서 Lab1→Lab2→Lab3를 이어서 실행**한다면 이 단계는 건너뛰어도 됩니다. 파일이 이미 Colab 작업 폴더에 남아있기 때문입니다.

> **Colab이 아니라 로컬(Jupyter, Anaconda 등)에서 실행 중**이라면 `google.colab` 모듈이 없어서 `ImportError`가 나는 게 정상입니다. 아래 코드는 이 경우를 자동으로 감지해서 안내 메시지만 출력하고 넘어갑니다.


In [14]:
try:
    from google.colab import files
    uploaded = files.upload()  # 방금 내려받은 data.npz 파일 선택
except ImportError:
    print("Colab이 아닌 환경입니다. data.npz 파일을 작업 폴더에 직접 복사해두세요.")


Colab이 아닌 환경입니다. data.npz 파일을 작업 폴더에 직접 복사해두세요.
